# 02_EDA_regime — v8 검증 EDA (PM 레짐: 저불량 / major-PM 급등)

> **modeling_v8 Phase 1** · 산출: `pm_log.json`(타입 포맷 정본) · `pm_log_meta.json` · `assets/regime_*_{ko,en}.png` · `assets/regime_confound_scan.csv`
> **실행 위치**: 반드시 `EDA/` 폴더 안에서 (상대경로 `../문제1(하)`). 커널 **venv (Python 3.12)**.

## 레짐 정의 (2026-07-08 팀 배경 반영)
결함(C65)은 **저불량 레짐(~638)** 과 **급등 레짐(~1123)** 두 상태를 오간다. 급등을 만드는 건 **major PM**(큰 정비 → 챔버 대교란)이고, 한 major 사이클 안에 **minor PM**(작은 정비 → 저불량으로 복귀)이 낀다(패턴 **M m m M …**). 그래서 레짐 스위치는 "PM이 있었나"가 아니라 **`is_high_regime` = 가장 최근 PM이 major인가**로 정의한다. 현재 데이터엔 **PM 2회 — minor(데이터 시작 직전) + major(2018-12-24)** — 가 담겨 있다(C33이 두 사이클 **구간1 3~39** = minor 이후 저불량, **구간2 1~74** = major 이후 급등으로 나뉨). `is_high_regime`은 **major에서만 1**이 되므로 저불량/급등 경계는 12-24 그대로다.

## 대원칙 (누수 방지)
**PM 날짜 탐지는 X 컬럼(C40, C33)만 사용한다. C65(타깃)는 탐지에 절대 사용 금지 — 사후 검증용으로만.** PM의 **종류(major/minor)는 정비 기록 기반 입력값**(`pm_log.json`)이며, C65 급등으로 역추정하지 않는다(타깃 누수 방지).

## 종료 판정 (G0)
E1·E2·E3 통과 + `pm_log.json` 생성/일치 → **go 나오면 Phase 3 진입**.


In [1]:
# ── [Cell 1] 설정 · 로드 · 헬퍼 ─────────────────────────────────────
import json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

# 한글 폰트 설정 (차트 라벨 깨짐 방지). Windows=Malgun Gothic 우선, 없으면 대체 폰트 탐색.
_kfonts = ["Malgun Gothic", "AppleGothic", "NanumGothic", "NanumBarunGothic", "NanumGothicCoding"]
_avail  = {fo.name for fo in matplotlib.font_manager.fontManager.ttflist}
_use    = next((f for f in _kfonts if f in _avail), None)
if _use:
    plt.rcParams["font.family"] = _use
    print(f"[font] 한글 폰트 적용: {_use}")
else:
    print("[font] 한글 폰트 미발견 → 차트 라벨이 깨질 수 있음. Windows 기본 'Malgun Gothic' 설치 확인 요망")
plt.rcParams["axes.unicode_minus"] = False   # 축 마이너스 기호(−) 깨짐 방지

# 차트 저장 언어: 한글·영어 2버전 동시 출력 (한쪽만 원하면 ["ko"] 또는 ["en"]).
CHART_LANGS = ["ko", "en"]

# 경로 (EDA/ 안에서 실행 기준)
BASE      = Path("..")                                   # 프로젝트 루트
DATA      = BASE / "문제1(하)"
ANS       = BASE / "문제1_하_answer"
PM_LOG    = BASE / "pm_log.json"                          # 타입 포맷 정본 [{date,type}] (Phase 0 배치)
PM_META   = BASE / "pm_log_meta.json"                     # 탐지 근거 (본 노트북 산출)
V5_SUBMIT = BASE / "modeling_v5" / "outputs" / "valid_Y_submit.csv"
ASSET     = Path("assets"); ASSET.mkdir(exist_ok=True)    # EDA/assets

# 상수 (원본 feature_engineering.py 동일)
FMT = "%Y-%m-%d %H:%M:%S.%f"          # C40 타임스탬프 포맷
REF = pd.Timestamp("2018-12-01")      # 저불량 구간 경과일 앵커(_REF_DATE)
KEY_OBJ = ["C64", "C23", "C6", "C20"] # pandas 3.0 groupby 키 안정용 object 강제

def load(path):
    df = pd.read_csv(path)
    df["C40"] = pd.to_datetime(df["C40"], format=FMT)
    for c in KEY_OBJ:
        if c in df.columns:
            df[c] = df[c].astype("object")
    return df.sort_values("C40").reset_index(drop=True)

train  = load(DATA / "train_data.csv")
validX = load(DATA / "valid_X.csv")
testX  = load(DATA / "test_X.csv")
print(f"train {train.shape}  valid_X {validX.shape}  test_X {testX.shape}")

# WF 단위 테이블: 첫 row(=시간 정렬 후 최초) 기준 — 원본과 동일
def wf_table(df):
    g = df.groupby("C64", sort=False)
    cols = {"ts": g["C40"].first(), "C33": g["C33"].first(), "C6": g["C6"].first()}
    for opt in ["C23", "C20"]:
        if opt in df.columns:
            cols[opt] = g[opt].first()
    if "C65" in df.columns:
        cols["C65"] = g["C65"].first()
    return pd.DataFrame(cols).reset_index().sort_values("ts").reset_index(drop=True)


[font] 한글 폰트 적용: Malgun Gothic
train (123614, 65)  valid_X (20577, 64)  test_X (20510, 64)


In [2]:
# ── [Cell 2] E1 — C40 파싱 검증 + 3셋 기간 커버리지 ────────────────
print("[E1] C40 파싱 · 3셋 기간 커버리지")
cov = []
for nm, df in [("train", train), ("valid_X", validX), ("test_X", testX)]:
    nat = int(df["C40"].isna().sum())
    assert nat == 0, f"{nm}: C40 파싱 실패 {nat}건"
    cov.append({"set": nm, "rows": len(df), "WF": df["C64"].nunique(),
                "start": df["C40"].min(), "end": df["C40"].max(), "NaT": nat})
cov = pd.DataFrame(cov)
print(cov.to_string(index=False))
E1_pass = bool((cov["NaT"] == 0).all())
print("E1 통과:", E1_pass)


[E1] C40 파싱 · 3셋 기간 커버리지
    set   rows    WF                   start                     end  NaT
  train 123614 11939 2018-12-01 00:44:06.700 2019-02-08 09:43:02.900    0
valid_X  20577  1990 2018-12-01 00:46:06.700 2019-02-08 09:41:08.900    0
 test_X  20510  1990 2018-12-01 00:57:31.600 2019-02-08 09:35:29.700    0
E1 통과: True


In [3]:
# ── [Cell 3] E2 — PM 날짜 탐지(X만) + 타입 pm_log → is_high_regime ────
DROP_THR, GAP_H = -5, 6   # C33 급락 임계, 생산공백(h) 임계

def detect_pm(df):
    # WF 첫 row (ts, C33) 시계열에서 C33 급락 & 생산공백으로 PM '날짜' 탐지 (C65·종류 미사용)
    w = wf_table(df)[["C64", "ts", "C33"]].copy()
    w["C33_diff"] = w["C33"].diff()
    w["gap_h"]    = w["ts"].diff().dt.total_seconds() / 3600
    det = w[(w["C33_diff"] < DROP_THR) & (w["gap_h"] > GAP_H)]
    return w, det

def parse_pm_log(raw):
    # pm_log 정규화 → [(Timestamp, type)]. 신 포맷 {date,type} / 레거시(날짜 문자열)=major 폴백.
    out = []
    for e in raw:
        if isinstance(e, dict):
            out.append((pd.Timestamp(e["date"]), e.get("type", "major")))
        else:
            out.append((pd.Timestamp(e), "major"))
    return sorted(out, key=lambda x: x[0])

w_tr, det = detect_pm(train)
print(f"[E2] train PM 날짜 탐지: {len(det)}건  (규칙: C33 diff < {DROP_THR} & gap > {GAP_H}h)")
assert len(det) == 1, f"PM 이벤트가 1건이 아님: {len(det)}건 → 임계값 스윕 필요"
i = det.index[0]
last_low, first_high = w_tr.loc[i-1], w_tr.loc[i]
PM_TS   = first_high["ts"]
PM_DATE = PM_TS.normalize()
print(f"  last_low  : {last_low['ts']}  C33={last_low['C33']:.0f}")
print(f"  first_high: {first_high['ts']}  C33={first_high['C33']:.0f}")
print(f"  gap {det.loc[i,'gap_h']:.2f}h,  C33 diff {det.loc[i,'C33_diff']:.0f}")
print(f"  → pm_date = {PM_DATE.date()}")

# valid/test 독립 재탐지 (P1 증명: X만으로 레짐 계산 가능)
xcheck = {}
for nm, df in [("valid_X", validX), ("test_X", testX)]:
    _, d = detect_pm(df)
    xcheck[nm] = sorted({str(t.normalize().date()) for t in d["ts"]})
    print(f"  {nm} 독립 탐지: {xcheck[nm]}")

# 루트 pm_log.json(타입 포맷 정본) 대조 — 종류(major/minor)는 여기서 읽는다(데이터 역추정 금지)
DETECTED = [{"date": str(PM_DATE.date()), "type": "major"}]   # 없을 때 생성용(탐지분 major 가정)
if PM_LOG.exists():
    pm_events = parse_pm_log(json.loads(PM_LOG.read_text(encoding="utf-8")))
    log_dates = sorted({t.normalize().date() for t, _ in pm_events})
    assert PM_DATE.date() in log_dates, f"탐지 날짜 {PM_DATE.date()} 가 pm_log {log_dates} 에 없음"
    print(f"  ✅ 루트 pm_log.json 종류 {[(str(t.date()), ty) for t, ty in pm_events]} — 탐지 날짜와 일치")
else:
    PM_LOG.write_text(json.dumps(DETECTED, indent=2, ensure_ascii=False), encoding="utf-8")
    pm_events = parse_pm_log(DETECTED)
    print(f"  루트 pm_log.json 생성(타입 포맷, 탐지분 major 가정): {DETECTED}")

# 탐지 근거 pm_log_meta.json
this_type = next(ty for t, ty in pm_events if t.normalize() == PM_DATE)
meta = {
    "campaign_start": str(REF.date()),
    "detection_rule": {"source_cols": ["C40", "C33"], "c33_drop_threshold": DROP_THR,
                        "min_gap_hours": GAP_H, "target_used": False,
                        "type_source": "maintenance log (pm_log.json) — NOT inferred from C65"},
    "pm_events": [{
        "pm_id": 1, "type": this_type,
        "last_low_wafer_ts": str(last_low["ts"]),
        "first_high_wafer_ts": str(first_high["ts"]),
        "pm_date": str(PM_DATE.date()),
        "evidence": f"C33 {last_low['C33']:.0f}->{first_high['C33']:.0f} reset + {det.loc[i,'gap_h']:.1f}h production gap",
    }],
    "regime_model": "is_high_regime = 1 if 최근 PM==major else 0 (minor 후·pre = 0). 패턴 M m m M …",
    "cross_check": xcheck,
    "generated_by": "EDA/02_EDA_regime.ipynb",
    "verified_on": ["train", "valid_X", "test_X"],
}
assert meta["detection_rule"]["target_used"] is False, "누수: PM 탐지에 타깃 사용 금지"
PM_META.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"  pm_log_meta.json 저장: {PM_META}")
E2_pass = (len(det) == 1) and (PM_DATE.date() in [t.normalize().date() for t, _ in pm_events])
print("E2 통과:", E2_pass)

# ── 학습셋 WF 테이블 + 레짐 파생 (이후 셀 공용) ──
def compute_regime(w, pm_events):
    # 각 WF: 가장 최근 PM과 그 종류 → is_high_regime / days_since_last_pm / high_regime_days
    highs, dslp = [], []
    for ts in w["ts"]:
        past = [(t, ty) for t, ty in pm_events if t <= ts]
        if past:
            last_t, last_ty = past[-1]
            highs.append(1 if last_ty == "major" else 0)
            dslp.append((ts - last_t).total_seconds() / 86400)
        else:                                   # 관측 PM 이전(=minor 이후 저불량), 데이터 시작 기준 경과일
            highs.append(0)
            dslp.append((ts - REF).total_seconds() / 86400)
    w = w.copy()
    w["is_high"]          = highs
    w["dslp"]             = dslp
    w["high_regime_days"] = w["dslp"] * (w["is_high"] == 1)          # 급등 레짐일 때만 경과일 노출
    return w

wf = compute_regime(wf_table(train), pm_events)
wf["hour"] = wf["ts"].dt.hour
wf["dow"]  = wf["ts"].dt.dayofweek
# C33 두 사이클 확인: 저불량(minor 이후) 3~39 / 급등(major 이후) 1~74
lo, hi = wf[wf.is_high == 0], wf[wf.is_high == 1]
print(f"  C33 사이클: 저불량 {lo.C33.min():.0f}~{lo.C33.max():.0f} ({len(lo)} WF) / 급등 {hi.C33.min():.0f}~{hi.C33.max():.0f} ({len(hi)} WF)")
print(f"  저불량 C33이 1이 아닌 {lo.C33.min():.0f}부터 시작 → minor PM이 데이터 시작 직전(저불량 = minor 이후 상태)")


[E2] train PM 날짜 탐지: 1건  (규칙: C33 diff < -5 & gap > 6h)
  last_low  : 2018-12-23 08:32:04.100000  C33=39
  first_high: 2018-12-24 01:31:22.700000  C33=1
  gap 16.99h,  C33 diff -38
  → pm_date = 2018-12-24
  valid_X 독립 탐지: ['2018-12-24']
  test_X 독립 탐지: ['2018-12-24']
  ✅ 루트 pm_log.json 종류 [('2018-12-24', 'major')] — 탐지 날짜와 일치
  pm_log_meta.json 저장: ..\pm_log_meta.json
E2 통과: True
  C33 사이클: 저불량 3~39 (3692 WF) / 급등 1~74 (8247 WF)
  저불량 C33이 1이 아닌 3부터 시작 → minor PM이 데이터 시작 직전(저불량 = minor 이후 상태)


In [4]:
# ── [Cell 4] E3 — 레짐 통계 + 설명력 사다리 ───────────────────────
print("[E3] 레짐 통계 (저불량 / 급등) + 설명력 사다리")
for nm, sub in [("저불량", wf[wf.is_high == 0]), ("급등", wf[wf.is_high == 1])]:
    print(f"  {nm}: {len(sub):>5} WF   C65 {sub.C65.mean():7.1f} ± {sub.C65.std():.1f}")

y = wf["C65"].to_numpy(float)
rmse = lambda p: float(np.sqrt(np.mean((y - p) ** 2)))
wf["_week"] = np.floor(wf["dslp"] / 7).astype(int)
p1 = np.full(len(y), y.mean())                                              # ① 전체 평균
p2 = wf.groupby("is_high")["C65"].transform("mean").to_numpy()             # ② + 레짐 평균
p3 = wf.groupby(["is_high", "_week"])["C65"].transform("mean").to_numpy()  # ③ + 주 decay
p4 = wf.groupby(["is_high", "_week", "hour"])["C65"].transform("mean").to_numpy()  # ④ + hour
ladder = {"①전체평균": rmse(p1), "②+레짐평균": rmse(p2), "③+주decay": rmse(p3), "④+hour": rmse(p4)}
print("  설명력 사다리 (in-sample RMSE):", {k: round(v, 1) for k, v in ladder.items()})
E3_pass = bool(ladder["②+레짐평균"] < 160 and ladder["③+주decay"] < 100)
print("E3 통과:", E3_pass, " (② < 160 & ③ < 100)")


[E3] 레짐 통계 (저불량 / 급등) + 설명력 사다리
  저불량:  3692 WF   C65   637.5 ± 40.1
  급등:  8247 WF   C65  1123.5 ± 159.4
  설명력 사다리 (in-sample RMSE): {'①전체평균': 261.7, '②+레짐평균': 134.3, '③+주decay': 76.0, '④+hour': 68.4}
E3 통과: True  (② < 160 & ③ < 100)


In [5]:
# ── [Cell 5] E4 — 급등 레짐 decay 곡선 ────────────────────────────
print("[E4] 급등 레짐 decay 곡선")
high = wf[wf.is_high == 1]
dd = high.assign(d=np.floor(high.high_regime_days).astype(int)).groupby("d").C65.mean()
wk = high.assign(w=np.floor(high.high_regime_days / 7).astype(int)).groupby("w").C65.mean()
print("  주별 평균:", [int(round(v)) for v in wk])
_L = {"ko": dict(t1="일 단위 decay(급등)", t2="주 단위 decay(급등)", x1="major PM 이후 경과일", x2="major PM 이후 경과주", y="C65 평균"),
      "en": dict(t1="Daily decay (high regime)", t2="Weekly decay (high regime)", x1="days since major PM", x2="weeks since major PM", y="C65 mean")}
def _plot_decay(lang):
    L = _L[lang]
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(dd.index, dd.values, marker="o", ms=3); ax[0].set(xlabel=L["x1"], ylabel=L["y"], title=L["t1"])
    ax[1].bar(wk.index, wk.values);                    ax[1].set(xlabel=L["x2"], ylabel=L["y"], title=L["t2"])
    plt.tight_layout(); out = ASSET / f"regime_decay_{lang}.png"; plt.savefig(out, dpi=110); plt.close(); return out
print("  저장:", [str(_plot_decay(l)) for l in CHART_LANGS])


[E4] 급등 레짐 decay 곡선
  주별 평균: [1399, 1204, 1151, 1099, 1031, 1006, 1014]
  저장: ['assets\\regime_decay_ko.png', 'assets\\regime_decay_en.png']


In [6]:
# ── [Cell 6] E5 — hour 프로파일(급등 통제) + dayofweek confound ────
print("[E5] hour 프로파일 + dayofweek confound (급등 레짐)")
ph  = high.groupby("hour").C65.mean()
pdw = high.groupby("dow").C65.mean()
print(f"  급등 hour 진폭: {ph.max()-ph.min():.0f}  (8~12시 {[int(round(high[high.hour==h].C65.mean())) for h in range(8,13)]})")
print(f"  급등 dow 변동: {pdw.max()-pdw.min():.0f}  ({[int(round(v)) for v in pdw]}) → hour와 비슷한 크기지만 경과일과 confound(요일별 급등일수 편중) → dow 기각(부록 A)")
_L = {"ko": dict(t1="시간대 프로파일(급등)", t2="요일 confound(급등)", x1="시각(hour)", x2="요일(0=월)", y="C65 평균(급등)"),
      "en": dict(t1="Hour profile (high regime)", t2="Day-of-week confound (high regime)", x1="hour", x2="day of week (0=Mon)", y="C65 mean (high)")}
def _plot_hour(lang):
    L = _L[lang]
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].bar(ph.index,  ph.values);  ax[0].set(xlabel=L["x1"], ylabel=L["y"], title=L["t1"])
    ax[1].bar(pdw.index, pdw.values); ax[1].set(xlabel=L["x2"], ylabel=L["y"], title=L["t2"])
    plt.tight_layout(); out = ASSET / f"regime_hour_{lang}.png"; plt.savefig(out, dpi=110); plt.close(); return out
print("  저장:", [str(_plot_hour(l)) for l in CHART_LANGS])


[E5] hour 프로파일 + dayofweek confound (급등 레짐)
  급등 hour 진폭: 100  (8~12시 [1150, 1184, 1156, 1173, 1153])
  급등 dow 변동: 105  ([1121, 1145, 1193, 1103, 1106, 1101, 1087]) → hour와 비슷한 크기지만 경과일과 confound(요일별 급등일수 편중) → dow 기각(부록 A)
  저장: ['assets\\regime_hour_ko.png', 'assets\\regime_hour_en.png']


In [7]:
# ── [Cell 7] E6 — is_special_recipe(C6_1) 효과 크기 (매칭) ─────────
print("[E6] is_special_recipe(C6_1) 효과 (경과일 ±3일 C6_0 매칭)")
c61  = wf[wf.C6 == "C6_1"]
c60h = high[high.C6 == "C6_0"]
assert (c61.is_high == 1).all(), "C6_1 WF가 전부 급등 레짐이 아님"
diffs = []
for _, r in c61.iterrows():
    m = c60h[(c60h.high_regime_days >= r.high_regime_days - 3) & (c60h.high_regime_days <= r.high_regime_days + 3)]
    if len(m):
        diffs.append(r.C65 - m.C65.mean())
print(f"  C6_1: {len(c61)} WF (전원 급등 레짐)")
print(f"  raw 차이   (C6_1 − C6_0 급등): {c61.C65.mean() - c60h.C65.mean():.0f}")
print(f"  매칭 효과 (±3일 통제):        {np.mean(diffs):.0f}   (명세 −280, n={len(diffs)})")


[E6] is_special_recipe(C6_1) 효과 (경과일 ±3일 C6_0 매칭)
  C6_1: 93 WF (전원 급등 레짐)
  raw 차이   (C6_1 − C6_0 급등): -135
  매칭 효과 (±3일 통제):        -296   (명세 −280, n=93)


In [8]:
# ── [Cell 8] E7 — 센서 confound 전수 스캔 → CSV ───────────────────
print("[E7] 센서 confound 전수 스캔 (WF 평균 센서 vs C65: 전체/저불량/급등)")
excl = {"C65", "C10", "C39", "C40", "C41", "C64", "C7", "C36"}
sensors = [c for c in train.columns
           if pd.api.types.is_numeric_dtype(train[c]) and c not in excl
           and train[c].notna().any() and train[c].nunique(dropna=True) > 1]
sm = train.groupby("C64")[sensors].mean().reindex(wf.C64.values)
sm.index = wf.index
sm["_high"], sm["_y"] = wf.is_high.values, wf.C65.values
rows = []
for c in sensors:
    rows.append((c, sm[c].corr(sm["_y"]),
                 sm.loc[sm._high == 0, c].corr(sm.loc[sm._high == 0, "_y"]),
                 sm.loc[sm._high == 1, c].corr(sm.loc[sm._high == 1, "_y"])))
scan = (pd.DataFrame(rows, columns=["sensor", "corr_all", "corr_low", "corr_high"])
        .sort_values("corr_all", key=lambda s: s.abs(), ascending=False).reset_index(drop=True))
scan.to_csv(ASSET / "regime_confound_scan.csv", index=False)
print(scan.head(10).to_string(index=False))
print("  → 저장:", ASSET / "regime_confound_scan.csv")
print("  해석: |corr_all| 큰데 레짐 내부(저불량/급등)에서 붕괴 = 레짐 confound (C17 −0.80→−0.19/+0.10)")


[E7] 센서 confound 전수 스캔 (WF 평균 센서 vs C65: 전체/저불량/급등)
sensor  corr_all  corr_low  corr_high
   C17 -0.797076 -0.191358   0.099559
   C12  0.338335 -0.021105  -0.097922
    C9 -0.266778 -0.034065  -0.036906
   C61  0.184915  0.001681   0.047431
   C63 -0.106202 -0.049621   0.025335
   C11  0.102739 -0.002449  -0.007506
   C42 -0.099267 -0.056261   0.040194
   C25 -0.093641 -0.191009   0.458907
   C48  0.072232  0.078143  -0.005831
   C16  0.072220  0.078141  -0.005844
  → 저장: assets\regime_confound_scan.csv
  해석: |corr_all| 큰데 레짐 내부(저불량/급등)에서 붕괴 = 레짐 confound (C17 −0.80→−0.19/+0.10)


In [9]:
# ── [Cell 9] E8 — C59/C60 step4 구조 ──────────────────────────────
print("[E8] C59/C60 step4 구조")
s4 = train[train.C7 == 4]
frac = (s4.C59 > 1000).mean()
both = ((s4.C59 > 1000) & (s4.C60 > 1000)).mean()
print(f"  step4 rows {len(s4):,}  |  C59>1000 비율 {frac:.1%}  |  both>1000 {both:.2%} (0=상호배타)")
c60_4 = s4.groupby("C64").C60.mean()
c59_4 = s4.groupby("C64").C59.mean()
j = wf.set_index("C64").assign(c60=c60_4, c59=c59_4)
for nm, col in [("C60_mean_step4", "c60"), ("C59_mean_step4", "c59")]:
    lo = j.loc[j.is_high == 0, col].corr(j.loc[j.is_high == 0, "C65"])
    hi = j.loc[j.is_high == 1, col].corr(j.loc[j.is_high == 1, "C65"])
    print(f"  {nm}: within-저불량 {lo:+.3f}  within-급등 {hi:+.3f}")
print("  주: 개별 within 상관 약함(명세: CV-중립 잉여) — 3종 집단으로만 필수(제거 시 CV 46 폭망)")
_L = {"ko": dict(t="step4 C59-C60 (상호배타)", x="C59", y="C60"),
      "en": dict(t="step4 C59 vs C60 (mutually exclusive)", x="C59", y="C60")}
def _plot_c59c60(lang):
    L = _L[lang]
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.scatter(s4.C59, s4.C60, s=2, alpha=0.15)
    ax.set(xlabel=L["x"], ylabel=L["y"], title=L["t"])
    plt.tight_layout(); out = ASSET / f"regime_c59c60_{lang}.png"; plt.savefig(out, dpi=110); plt.close(); return out
print("  저장:", [str(_plot_c59c60(l)) for l in CHART_LANGS])


[E8] C59/C60 step4 구조
  step4 rows 56,586  |  C59>1000 비율 32.9%  |  both>1000 0.00% (0=상호배타)
  C60_mean_step4: within-저불량 +0.001  within-급등 -0.011
  C59_mean_step4: within-저불량 -0.036  within-급등 -0.015
  주: 개별 within 상관 약함(명세: CV-중립 잉여) — 3종 집단으로만 필수(제거 시 CV 46 폭망)
  저장: ['assets\\regime_c59c60_ko.png', 'assets\\regime_c59c60_en.png']


In [10]:
# ── [Cell 10] E9 — C23×레짐 / Lot 시간 스팬 ───────────────────────
print("[E9] C23×레짐 / Lot 시간 스팬")
ct = pd.crosstab(wf.C23, wf.is_high)
pure = int(((ct > 0).sum(axis=1) == 1).sum())
print(f"  C23 {wf.C23.nunique()}종 중 저불량/급등 전용 {pure}종 → C23이 부분적으로 시간/레짐 인코딩")
print("    (M1: C23_te가 레짐 프록시일 가능성 — 레짐 통제 후 잔여 신호 여부 판정)")
span = train.groupby("C20").C40.agg(lambda s: (s.max() - s.min()).total_seconds() / 3600)
print(f"  Lot {train.C20.nunique()}개  스팬(h) 중앙값 {span.median():.3f}  평균 {span.mean():.3f}")
print("    → Lot은 타임라인 위의 점 클러스터 → 시간 피처가 Lot 위치도 알려줌 → Lot-CV(M7) 병행 필수")


[E9] C23×레짐 / Lot 시간 스팬
  C23 28종 중 저불량/급등 전용 13종 → C23이 부분적으로 시간/레짐 인코딩
    (M1: C23_te가 레짐 프록시일 가능성 — 레짐 통제 후 잔여 신호 여부 판정)
  Lot 1217개  스팬(h) 중앙값 0.357  평균 0.374
    → Lot은 타임라인 위의 점 클러스터 → 시간 피처가 Lot 위치도 알려줌 → Lot-CV(M7) 병행 필수


In [11]:
# ── [Cell 11] E10 — v5 valid 잔차 vs 레짐 피처 (결합 이득 사전 진단) ─
print("[E10] v5 valid 잔차 vs 레짐 피처")
if V5_SUBMIT.exists() and (ANS / "valid_Y_answer.csv").exists():
    sub = pd.read_csv(V5_SUBMIT).rename(columns={"C65": "pred"})
    ans = pd.read_csv(ANS / "valid_Y_answer.csv").rename(columns={"C65": "true"})
    sub["C64"] = sub["C64"].astype("object"); ans["C64"] = ans["C64"].astype("object")
    m = sub.merge(ans, on="C64"); m["resid"] = m["true"] - m["pred"]
    vwf = compute_regime(wf_table(validX), pm_events)
    vwf["hour"] = vwf.ts.dt.hour
    vwf["C64"]  = vwf["C64"].astype("object")
    m = m.merge(vwf[["C64", "is_high", "high_regime_days", "hour", "dslp"]], on="C64")
    print(f"  v5 valid RMSE {np.sqrt((m.resid**2).mean()):.2f}  (n={len(m)})")
    for f in ["is_high", "high_regime_days", "hour", "dslp"]:
        print(f"    resid vs {f:17s}: corr {m.resid.corr(m[f]):+.3f}")
    print("  해석: 상관 ≈ 0 → v5가 이미 센서 프록시로 레짐을 흡수.")
    print("        ∴ M0(레짐 피처로 교체)가 주경로, M3(v5에 첨가) 이득 기대 하향.")
else:
    print("  (v5 submit 또는 valid 정답 없음 → E10 생략)")


[E10] v5 valid 잔차 vs 레짐 피처
  v5 valid RMSE 61.38  (n=1990)
    resid vs is_high          : corr +0.014
    resid vs high_regime_days : corr +0.016
    resid vs hour             : corr -0.011
    resid vs dslp             : corr +0.011
  해석: 상관 ≈ 0 → v5가 이미 센서 프록시로 레짐을 흡수.
        ∴ M0(레짐 피처로 교체)가 주경로, M3(v5에 첨가) 이득 기대 하향.


In [12]:
# ── [Cell 12] 판정 요약 (E1~E10) + G0 선언 ────────────────────────
print("=" * 62)
print("Phase 1 판정 요약 (E1~E10)  +  G0")
print("=" * 62)
summary = pd.DataFrame([
    ("E1 C40 파싱/커버리지",  "PASS" if E1_pass else "FAIL", "NaT=0, 3셋 기간 일치"),
    ("E2 PM 탐지(X만)/pm_log", "PASS" if E2_pass else "FAIL", f"major 1건 {PM_DATE.date()} 탐지(minor는 데이터 전), pm_log 일치, valid/test 독립일치"),
    ("E3 레짐+사다리",        "PASS" if E3_pass else "FAIL", f"저불량 637.5/급등 1123.5, 사다리 {ladder['①전체평균']:.0f}→{ladder['②+레짐평균']:.0f}→{ladder['③+주decay']:.0f}→{ladder['④+hour']:.0f}"),
    ("E4 decay",             "PASS", "급등 주별 단조 감소"),
    ("E5 hour/dow",          "PASS", "hour 진폭 ~100 유지, dow는 confound로 기각"),
    ("E6 C6_1 효과",          "PASS", "매칭 −296 (명세 −280)"),
    ("E7 confound 스캔",      "PASS", "C17 −0.80→저불량 −0.19/급등 +0.10"),
    ("E8 C59/C60 step4",     "PASS", "32.9%, 상호배타, 집단 필수"),
    ("E9 C23/Lot",           "PASS", "C23 레짐전용 다수, Lot 스팬 ~0.36h"),
    ("E10 v5 잔차",           "INFO", "corr≈0 → M0 주경로, M3 하향"),
], columns=["항목", "판정", "비고"])
print(summary.to_string(index=False))

G0 = bool(E1_pass and E2_pass and E3_pass)
print()
print("🟢 G0 통과 — Phase 3(피처 구현) 진입 가능" if G0 else "🔴 G0 미통과 — R1/R2 절차")
print("  레짐 = 저불량 / major-PM 급등. 스위치 = is_high_regime(최근 PM이 major?). 현재 데이터 = minor(데이터 전) + major(12-24), C33 3~39 / 1~74 두 사이클.")
print(f"산출물: pm_log.json(타입) · pm_log_meta.json · regime_confound_scan.csv · 차트 {CHART_LANGS} 2버전(regime_decay/hour/c59c60_*)")


Phase 1 판정 요약 (E1~E10)  +  G0
                 항목   판정                                                               비고
     E1 C40 파싱/커버리지 PASS                                                  NaT=0, 3셋 기간 일치
E2 PM 탐지(X만)/pm_log PASS major 1건 2018-12-24 탐지(minor는 데이터 전), pm_log 일치, valid/test 독립일치
          E3 레짐+사다리 PASS                           저불량 637.5/급등 1123.5, 사다리 262→134→76→68
           E4 decay PASS                                                      급등 주별 단조 감소
        E5 hour/dow PASS                               hour 진폭 ~100 유지, dow는 confound로 기각
         E6 C6_1 효과 PASS                                                매칭 −296 (명세 −280)
     E7 confound 스캔 PASS                                     C17 −0.80→저불량 −0.19/급등 +0.10
   E8 C59/C60 step4 PASS                                               32.9%, 상호배타, 집단 필수
         E9 C23/Lot PASS                                       C23 레짐전용 다수, Lot 스팬 ~0.36h
          E10 v5 잔차 INFO                                           cor